[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/36_explainable_ai/01_global_explanations.ipynb)

# Bài 01 — Giải thích toàn cục

**🎯 Mục tiêu**
- Vì sao cần giải thích mô hình; phân loại: **toàn cục** vs **cục bộ**, **gắn với mô hình** vs **không phụ thuộc mô hình**; mô hình **vốn dễ hiểu**.
- Đặt cạnh nhau "sự thật" (dữ liệu có logit thật đã biết) và các giải thích — để biết giải thích nào **đúng**.
- **Độ quan trọng đặc trưng**: impurity (MDI) vs **permutation importance** — và thiên lệch của MDI.
- **PDP, ICE, ALE**: đặc trưng ảnh hưởng dự đoán theo hình dạng nào; vì sao PDP sai khi đặc trưng tương quan.
- **Mô hình thay thế** (global surrogate): cây quyết định nhỏ bắt chước mô hình đen — và độ trung thực.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/36_explainable_ai

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, export_text

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()

## 1. Vì sao cần giải thích?

- **Gỡ lỗi**: mô hình có dựa vào đặc trưng rò rỉ, đường tắt (chương 33), lỗi dữ liệu không?
- **Niềm tin và chấp nhận**: bác sĩ, nhân viên tín dụng cần biết vì sao trước khi làm theo.
- **Tuân thủ**: quyền được giải thích quyết định tự động (GDPR, quy định tín dụng); kiểm toán công bằng.
- **Khám phá**: mô hình học được quy luật gì từ dữ liệu (cẩn thận: mô hình ≠ thế giới thật — bài 03).

| | Toàn cục (bài này) | Cục bộ (bài 02) |
|---|---|---|
| Câu hỏi | Mô hình nói chung dựa vào gì? | Vì sao dự đoán **này** như vậy? |
| Công cụ | importance, PDP/ALE, surrogate | LIME, SHAP, counterfactual |

Phương pháp **không phụ thuộc mô hình** (model-agnostic) chỉ cần gọi `predict` — dùng được cho mọi mô hình; phương pháp **gắn với mô hình** (hệ số hồi quy, MDI của cây) dùng cấu trúc bên trong.

## 2. Dữ liệu có "đáp án"

Khách hàng viễn thông rời bỏ (chương 7), sinh từ một **logit thật đã biết** — nên ta biết đặc trưng nào thật sự ảnh hưởng và theo hướng nào:

$$\text{logit} = -1.6 + 1.3\cdot\text{month\_to\_month} - 0.5\cdot\text{two\_year} - 0.035\cdot\text{tenure} + 0.018\cdot(\text{monthly} - 70) + 0.38\cdot\text{support\_calls} + 0.45\cdot\text{fiber} + 0.35\cdot\text{ewallet} + 0.3\cdot\text{senior} - 0.2\cdot(\text{n\_products} - 1)$$

Hai đặc trưng **không** có trong logit: `total` (= tenure × monthly — tương quan mạnh với cả hai) và `noise_id` (mã ngẫu nhiên, nhiều giá trị khác nhau).

In [ ]:
raw = make_customer_churn(20000, seed=0)
X = pd.DataFrame({
    "tenure": raw.tenure_months, "monthly": raw.monthly_charges, "total": raw.total_charges,
    "support_calls": raw.support_calls, "month_to_month": (raw.contract == "month-to-month").astype(int),
    "two_year": (raw.contract == "two_year").astype(int), "fiber": (raw.internet_service == "fiber").astype(int),
    "ewallet": (raw.payment_method == "e-wallet").astype(int), "senior": raw.senior.astype(int),
    "n_products": raw.n_products,
})
X["noise_id"] = np.random.default_rng(0).permutation(len(X))           # nhiễu thuần, nhiều giá trị
X = X.astype(float)
y = raw.churn.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
true_logit = lambda D: (-1.6 + 1.3 * D.month_to_month - 0.5 * D.two_year - 0.035 * D.tenure + 0.018 * (D.monthly - 70)
                        + 0.38 * D.support_calls + 0.45 * D.fiber + 0.35 * D.ewallet + 0.3 * D.senior - 0.2 * (D.n_products - 1))
print(f"{len(X):,} khách hàng, tỉ lệ rời bỏ {y.mean():.1%} | corr(total, tenure) = {X.total.corr(X.tenure):.2f}")

## 3. Mô hình vốn dễ hiểu vs mô hình "hộp đen"

In [ ]:
scaler = StandardScaler().fit(X_train)
logreg = LogisticRegression(max_iter=2000).fit(scaler.transform(X_train), y_train)
forest = RandomForestClassifier(300, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(X_train, y_train)
boosting = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(X_train, y_train)

for name, model, Xt in [("Logistic regression", logreg, scaler.transform(X_test)), ("Random forest", forest, X_test),
                        ("Gradient boosting", boosting, X_test)]:
    print(f"{name:20s}: ROC-AUC = {roc_auc_score(y_test, model.predict_proba(Xt)[:, 1]):.4f}")
print(f"{'logit thật (trần)':20s}: ROC-AUC = {roc_auc_score(y_test, true_logit(X_test)):.4f}")

coef = pd.Series(logreg.coef_[0] / scaler.scale_, index=X.columns)     # đổi về thang gốc của đặc trưng
truth = pd.Series({"tenure": -0.035, "monthly": 0.018, "total": 0, "support_calls": 0.38, "month_to_month": 1.3, "two_year": -0.5,
                   "fiber": 0.45, "ewallet": 0.35, "senior": 0.3, "n_products": -0.2, "noise_id": 0})
print(pd.DataFrame({"hệ số logistic (thang gốc)": coef.round(4), "hệ số thật": truth}))

Dữ liệu này **cộng tính trong không gian logit** — đúng giả định của logistic regression — nên mô hình đơn giản nhất lại tốt nhất (sát trần của logit thật), và **hệ số của nó chính là lời giải thích**, khớp gần đúng với sự thật (`total`, `noise_id` ≈ 0). Bài học (Rudin, 2019): với dữ liệu bảng, trước khi giải thích một hộp đen, hãy thử mô hình dễ hiểu (hồi quy, GAM, cây nông) — nếu tốt ngang nhau, dùng nó.

Phần còn lại dùng **gradient boosting** làm "hộp đen" để xem các phương pháp giải thích có tìm lại được sự thật không.

## 4. Độ quan trọng đặc trưng: MDI vs permutation

- **MDI** (mean decrease in impurity — `feature_importances_` của cây/rừng): tổng mức giảm impurity tại các nút tách theo đặc trưng đó, tính **trên tập train**.
- **Permutation importance**: xáo trộn một cột ở **tập test**, đo hiệu năng giảm bao nhiêu (chương 11). Không phụ thuộc mô hình.

In [ ]:
t0 = time.perf_counter()
perm_rf = permutation_importance(forest, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=-1)
perm_gb = permutation_importance(boosting, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0)
effect_size = truth.abs() * X.std()                                    # độ lớn tác động thật: |β| × độ lệch chuẩn của đặc trưng
importance = pd.DataFrame({"MDI (rừng)": forest.feature_importances_, "permutation (rừng)": perm_rf.importances_mean,
                           "permutation (boosting)": perm_gb.importances_mean, "tác động thật |β|·std": effect_size}, index=X.columns)
print(importance.round(4).sort_values("permutation (boosting)", ascending=False))
print(f"{time.perf_counter() - t0:.0f}s")

- **MDI xếp `noise_id` (nhiễu thuần) lên hạng cao**, ngang các đặc trưng thật: đặc trưng nhiều giá trị khác nhau có nhiều điểm tách khả dĩ, cây sâu dùng chúng để "học thuộc" tập train, và MDI đo trên tập train nên ghi công cho việc đó (Strobl et al., 2007). Permutation importance trên tập **test** cho nó ≈ 0 — đúng.
- `total` có MDI lớn nhưng permutation ≈ 0: thông tin của nó trùng với `tenure` và `monthly`; xáo trộn riêng `total` thì mô hình vẫn còn hai đặc trưng kia. (Điều ngược lại cũng xảy ra: hai đặc trưng tương quan "chia nhau" độ quan trọng — bài 03.)
- Thứ tự permutation importance của boosting khớp khá tốt với độ lớn tác động thật |β|·std (month-to-month, monthly, tenure, support_calls đứng đầu; các đặc trưng tác động nhỏ xếp cuối).

Quy tắc: tránh MDI để kết luận; dùng permutation importance trên **dữ liệu chưa thấy**, nhiều lần lặp, kèm độ lệch chuẩn.

## 5. PDP và ICE: đặc trưng ảnh hưởng theo hình dạng nào?

**Partial dependence** của đặc trưng $j$: đặt $x_j = v$ cho **mọi** mẫu, lấy trung bình dự đoán: $\text{PD}_j(v) = \frac{1}{n}\sum_i f(v, x_{i, -j})$. **ICE** (individual conditional expectation): vẽ riêng đường $f(v, x_{i,-j})$ của từng mẫu — PDP là trung bình các đường ICE. ICE song song → hiệu ứng như nhau cho mọi người; ICE cắt nhau → có **tương tác**.

In [ ]:
X_show = X_test.sample(400, random_state=0)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
PartialDependenceDisplay.from_estimator(boosting, X_show, ["tenure", "monthly", "support_calls"], kind="both", ax=axes,
                                        ice_lines_kw={"alpha": 0.08, "color": "C0"}, pd_line_kw={"color": "C3", "lw": 2.5},
                                        response_method="predict_proba", method="brute", random_state=0)
plt.tight_layout()
plt.show()

Hướng tác động khớp logit thật: rời bỏ giảm theo thời gian gắn bó, tăng theo cước tháng và số cuộc gọi hỗ trợ. Các đường ICE gần như song song (sai khác chủ yếu do mức nền khác nhau) — đúng với dữ liệu không có tương tác trong logit; độ "gãy" nhỏ trên các đường là do boosting xấp xỉ một hàm trơn bằng các bậc thang.

## 6. Khi đặc trưng tương quan: PDP ngoại suy, ALE thì không

PDP đặt `total` = v cho **mọi** khách hàng — kể cả khách mới 1 tháng tuổi với `total` = 8 000 (không thể có, vì total ≈ tenure × monthly). Mô hình bị hỏi ở vùng **không có dữ liệu** → câu trả lời là ngoại suy tuỳ tiện. **ALE** (Accumulated Local Effects, Apley & Zhu, 2020) chỉ so sánh dự đoán **trong từng khoảng nhỏ** của đặc trưng, trên các mẫu **thật sự nằm trong khoảng đó**, rồi cộng dồn các hiệu nhỏ — không bao giờ tạo tổ hợp phi thực tế.

In [ ]:
def ale_1d(predict, X, feature, n_bins=20):
    """ALE bậc 1 (trên thang logit). Trả về (biên các khoảng, hiệu ứng đã căn giữa)."""
    edges = np.unique(np.quantile(X[feature], np.linspace(0, 1, n_bins + 1)))
    bin_idx = np.clip(np.searchsorted(edges, X[feature], side="right") - 1, 0, len(edges) - 2)
    local = np.zeros(len(edges) - 1)
    counts = np.zeros(len(edges) - 1)
    for k in range(len(edges) - 1):
        in_bin = X[bin_idx == k]
        if len(in_bin) == 0:
            continue
        lo, hi = in_bin.copy(), in_bin.copy()
        lo[feature], hi[feature] = edges[k], edges[k + 1]
        local[k] = (predict(hi) - predict(lo)).mean()                   # hiệu cục bộ, chỉ trên mẫu trong khoảng
        counts[k] = len(in_bin)
    acc = np.r_[0, np.cumsum(local)]
    centers = (acc[:-1] + acc[1:]) / 2
    return edges, acc - np.average(centers, weights=counts)


def logit_of(model):
    return lambda D: model.decision_function(D) if hasattr(model, "decision_function") else np.log(model.predict_proba(D)[:, 1] / (1 - model.predict_proba(D)[:, 1]))


def pdp_1d(predict, X, feature, grid):
    return np.array([predict(X.assign(**{feature: v})).mean() for v in grid])


f_gb = logit_of(boosting)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, feat in zip(axes, ["tenure", "total"]):
    edges, ale = ale_1d(f_gb, X_test, feat)
    pdp = pdp_1d(f_gb, X_test, feat, edges)
    ax.plot(edges, pdp - pdp.mean(), "o-", label="PDP (căn giữa)")
    ax.plot(edges, ale, "s-", label="ALE")
    if feat == "tenure":
        ax.plot(edges, -0.035 * (edges - edges.mean()), "k:", label="hiệu ứng thật (−0.035/tháng)")
    else:
        ax.axhline(0, color="k", ls=":", label="hiệu ứng thật (0)")
    ax.set_xlabel(feat)
    ax.set_ylabel("ảnh hưởng lên logit")
    ax.legend(fontsize=8)
    print(f"{feat:7s}: biên độ PDP = {np.ptp(pdp):.2f} | biên độ ALE = {np.ptp(ale):.2f} (logit)")
plt.tight_layout()
plt.show()

Hai quan sát:
- Với `tenure`, ALE bám đường thật sát hơn PDP. PDP bị "làm phẳng" vì lấy trung bình cả dự đoán ở những tổ hợp không tồn tại (khách gắn bó 70 tháng mà tổng cước rất nhỏ) — nơi cây chỉ ngoại suy.
- Cả hai đều cho `total` một tác động âm, dù tác động thật bằng 0, và cả hai đều đánh giá thấp tác động thật của `tenure`. Lý do: `total` tương quan mạnh với `tenure`, nên mô hình đã "chia" một phần tác động của tenure sang total. ALE và PDP mô tả trung thực **mô hình** — và mô hình không giống quá trình sinh dữ liệu.

Bài học: khi đặc trưng tương quan, ưu tiên ALE (không ngoại suy), nhưng nhớ rằng mọi phương pháp chỉ giải thích **mô hình**. Câu hỏi nhân quả "tăng total thì churn có giảm không" cần phương pháp của chương 38 (bài 03 quay lại vấn đề tương quan).

## 7. Mô hình thay thế toàn cục (global surrogate)

Huấn luyện một mô hình **dễ hiểu** (cây nông) để bắt chước **dự đoán** của mô hình đen; đo **độ trung thực** ($R^2$ giữa dự đoán của hai mô hình) — không phải độ chính xác so với nhãn thật.

In [ ]:
p_black_box = boosting.predict_proba(X_test)[:, 1]
for depth in [2, 3, 5]:
    surrogate = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_test, p_black_box)
    print(f"cây sâu {depth}: độ trung thực R² = {surrogate.score(X_test, p_black_box):.3f}")
surrogate3 = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_test, p_black_box)
print(export_text(surrogate3, feature_names=list(X.columns), decimals=2))

Cây sâu 3 giải thích hơn 80% biến thiên trong dự đoán của boosting bằng vài luật đọc được: hợp đồng theo tháng → cước tháng → tenure (với hợp đồng dài hạn) hoặc số cuộc gọi hỗ trợ (với hợp đồng theo tháng). Cây sâu hơn trung thực hơn nhưng khó đọc hơn — đánh đổi trung thực ↔ dễ hiểu. Luôn báo cáo độ trung thực kèm theo; một surrogate trung thực 50% thì "giải thích" của nó sai một nửa.

## 🧠 Kiểm tra nhanh

1. Phân biệt giải thích toàn cục/cục bộ, gắn với mô hình/không phụ thuộc mô hình.
2. Vì sao MDI đánh giá cao đặc trưng nhiễu nhiều giá trị? Permutation importance khắc phục thế nào?
3. PDP và ICE khác nhau ra sao? ICE cắt nhau nói lên điều gì?
4. Vì sao PDP sai khi đặc trưng tương quan? ALE làm gì khác?
5. Global surrogate là gì? Vì sao phải báo cáo độ trung thực?

## 📌 Tóm tắt

- Thử mô hình dễ hiểu trước; với dữ liệu cộng tính, logistic regression vừa tốt nhất vừa tự giải thích.
- MDI thiên lệch (đặc trưng nhiều giá trị, tính trên train) → dùng permutation importance trên dữ liệu chưa thấy.
- PDP/ICE cho hình dạng ảnh hưởng; ALE an toàn hơn khi đặc trưng tương quan.
- Surrogate dễ đọc nhưng phải đi kèm độ trung thực. Mọi phương pháp giải thích **mô hình**, không giải thích **quan hệ nhân quả** trong thế giới.

## 📚 Tài liệu tham khảo

- Molnar — *Interpretable Machine Learning* (sách mở, [christophm.github.io/interpretable-ml-book](https://christophm.github.io/interpretable-ml-book/))
- Rudin (2019) — *Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead*
- Strobl et al. (2007) — *Bias in random forest variable importance measures*
- Friedman (2001) — *Greedy function approximation: A gradient boosting machine* (PDP); Goldstein et al. (2015) — *Peeking Inside the Black Box* (ICE)
- Apley & Zhu (2020) — *Visualizing the effects of predictor variables in black box supervised learning models* (ALE)
- Fisher, Rudin & Dominici (2019) — *All Models are Wrong, but Many are Useful* (model reliance, permutation importance)